# 04 预测任务设计：时间划分与联合窗口

本阶段只检查可用性，不训练、不调参、不计算测试集预测成绩。
固定候选划分：训练为源日期2020-06-01至2023-12-31，验证为2024全年，测试为2025-01-01至2026-06-30。
小时标签24转为次日00:00，因此分界采用右闭区间：训练截至2024-01-01 00:00，验证截至2025-01-01 00:00。
划分在分析前固定，后续只用训练/验证选择方案；测试集分布仅用于报告覆盖，不用于选模型。

72小时输入、24小时目标。每个预测目标必须全部落入同一划分；验证/测试允许使用预测时点之前已观测到的历史上下文。
采用滚动预测情景，不是在测试期开始时一次预测整个测试期。所有结果均非最终模型选择。


In [8]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src'/'aqhi_data.py').exists(): ROOT=ROOT.parent
candidates=[]
for p in sorted((ROOT/'data/processed/full_history').iterdir()):
    if (p/'summary.json').exists() and (p/'history_all_audited.csv.gz').exists():
        s=json.loads((p/'summary.json').read_text(encoding='utf-8'))
        if s['source_month_start']=='201312' and s['source_month_end']=='202606': candidates.append(p)
assert candidates
SOURCE=candidates[-1]
h=pd.read_csv(SOURCE/'history_all_audited.csv.gz',keep_default_na=False,dtype={'source_date':str})
h=h.loc[h.source_date.ge('2020-06-01')].copy()
h['timestamp']=pd.to_datetime(h.timestamp,utc=True).dt.tz_convert('Asia/Hong_Kong')
h['aqhi_value']=pd.to_numeric(h.aqhi_value,errors='coerce')
for c in ['is_substituted','is_10_plus','is_missing','is_unavailable','is_invalid']:
    if h[c].dtype!=bool:
        assert h[c].astype(str).isin(['True','False']).all()
        h[c]=h[c].astype(str).eq('True')
assert not h.duplicated(['station_id','timestamp']).any()
tz='Asia/Hong_Kong'
train_end=pd.Timestamp('2024-01-01',tz=tz)
val_end=pd.Timestamp('2025-01-01',tz=tz)
h['split']=np.where(h.timestamp<=train_end,'train',np.where(h.timestamp<=val_end,'validation','test'))
L,H=72,24
idx=pd.date_range(h.timestamp.min(),h.timestamp.max(),freq='h')
OUT=ROOT/'data/processed/design'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT.mkdir(parents=True,exist_ok=False)
print('输入：',SOURCE)
print('输出：',OUT)
display(h.groupby('split').agg(first=('timestamp','min'),last=('timestamp','max'),rows=('timestamp','size')))

输入： C:\Users\hxq08259\Desktop\sem1\Math7005\data\processed\full_history\20261110_170756_651230
输出： C:\Users\hxq08259\Desktop\sem1\Math7005\data\processed\design\20261110_180616_753696


,first,last,rows
split,,,
test,2025-01-01 01:00:00+08:00,2026-07-01 00:00:00+08:00,235872
train,2020-06-01 01:00:00+08:00,2024-01-01 00:00:00+08:00,562464
validation,2024-01-01 01:00:00+08:00,2025-01-01 00:00:00+08:00,158112


## 1. 10+ 与目标类别的覆盖

10+保留为截尾类别，不当作数值11。候选风险分类保留原解析器的类别。
统计分母是已有行，缺行单列处理。分类方案能保留10+，但仍需解决输入编码和类别不平衡。
不能仅凭少量高类别样本或整体准确率宣布模型有效。

In [2]:
coverage=h.groupby(['split','station_id']).agg(rows=('timestamp','size'),
    exact_values=('aqhi_value','count'),capped_10_plus=('is_10_plus','sum'),
    substituted=('is_substituted','sum'),blank=('is_missing','sum'),unavailable=('is_unavailable','sum'))
coverage['capped_pct']=100*coverage.capped_10_plus/coverage.rows
risk=h.risk_category.replace('','Unknown')
risk_counts=pd.crosstab([h.split,h.station_id],risk).reindex(
    columns=['Low','Moderate','High','Very High','Serious','Unknown'],fill_value=0)
display(coverage.round(3))
display(risk_counts.groupby(level=0).sum())

rows  exact_values  capped_10_plus  substituted  \
split      station_id                                                          
test       Causeway Bay     13104         13082               4          333   
           Central          13104         13068               1          437   
           Central/Western  13104         13084               3          274   
           Eastern          13104         13085               2          312   
           Kwai Chung       13104         13085               2          286   
           Kwun Tong        13104         13084               3          299   
           Mong Kok         13104         13085               2          209   
           North            13104         13080               4          601   
           Sha Tin          13104         13082               2          354   
           Sham Shui Po     13104         13086               1          335   
           Southern         13104         13087               0          164   
           Tai Po           13104         13080               4          357   
           Tap Mun          13104         12797               0          414   
           Tseung Kwan O    13104         13083               4          232   
           Tsuen Wan        13104         13087               0          242   
           Tuen Mun         13104         13071              15         1212   
           Tung Chung       13104         13080               7          364   
           Yuen Long        13104         13086               1          311   
train      Causeway Bay     31248         30749               6          684   
           Central          31248         30750               6          796   
           Central/Western  31248         30726              31          589   
           Eastern          31248         30747              10          768   
           Kwai Chung       31248         30745              12          558   
           Kwun Tong        31248         30743              14          571   
           Mong Kok         31248         30747               7          805   
           North            31248         29795              20         1143   
           Sha Tin          31248         30755               1          614   
           Sham Shui Po     31248         30742              15         1149   
           Southern         31248         29799              17          956   
           Tai Po           31248         30749               7          818   
           Tap Mun          31248         29425              37          904   
           Tseung Kwan O    31248         30748               9          825   
           Tsuen Wan        31248         30745              12          747   
           Tuen Mun         31248         30715              41         1196   
           Tung Chung       31248         30734              22          833   
           Yuen Long        31248         30722              34          891   
validation Causeway Bay      8784          8783               0          118   
           Central           8784          8783               0          186   
           Central/Western   8784          8782               1          124   
           Eastern           8784          8783               0          158   
           Kwai Chung        8784          8783               0          157   
           Kwun Tong         8784          8783               0          194   
           Mong Kok          8784          8783               0          212   
           North             8784          8783               0          235   
           Sha Tin           8784          8783               0          186   
           Sham Shui Po      8784          8783               0          162   
           Southern          8784          8783               0          213   
           Tai Po            8784          8783               0          164   
           Tap Mun           

risk_category,Low,Moderate,High,Very High,Serious,Unknown
split,,,,,,
test,142069,89982,1966,1175,55,625
train,328253,212409,5194,4280,301,12027
validation,97857,58155,1324,630,14,132


## 2. 联合窗口：两种可用性定义

**exact_only**：输入与目标都要求1–10精确数值，是保守回归可用性检查，会排除10+。
**category_observed**：允许1–10及10+，但不允许缺行/缺值。只是分类方案的可用性上界，尚未编码或训练。
两者均暂时保留带星号的观测。

对比18站输入、17站输入（不含塔门）、单站输入。
为了公平，所有输入方案都预测相同的17个非塔门目标站。塔门仍保留在原始数据中，不代表永久放弃它。
先统计每个目标站的窗口，再统计同一时点能同时预测全部17站的窗口。


In [3]:
values=h.pivot(index='timestamp',columns='station_id',values='aqhi_value').reindex(idx)
capped=h.pivot(index='timestamp',columns='station_id',values='is_10_plus').reindex(idx).fillna(False).astype(bool)
all_stations=sorted(values.columns)
assert len(all_stations)==18
targets=[s for s in all_stations if s!='Tap Mun']
panels={'exact_only':values.notna(),'category_observed':values.notna()|capped}
groups={'all18':all_stations,'without_tap_mun17':targets}
target_start=pd.Series(idx,index=idx)
target_end=target_start+pd.Timedelta(hours=H-1)
split_masks={'train':target_end.le(train_end),
 'validation':target_start.gt(train_end)&target_end.le(val_end),
 'test':target_start.gt(val_end)&target_end.le(idx.max())}
def past_ok(series):
    return series.astype(int).rolling(L,min_periods=L).sum().shift(1).eq(L)
def future_ok(series):
    return series.astype(int).rolling(H,min_periods=H).sum().shift(-(H-1)).eq(H)
rows=[]; joint_rows=[]; masks={}
for policy,panel in panels.items():
    for group,names in groups.items():
        inputs=past_ok(panel[names].all(axis=1))
        common=inputs.copy()
        for station in targets:
            valid=inputs & future_ok(panel[station])
            masks[(policy,group,station)]=valid
            common &= future_ok(panel[station])
            for split,sm in split_masks.items():
                rows.append({'policy':policy,'inputs':group,'target':station,'split':split,'windows':int((valid&sm).sum())})
        for split,sm in split_masks.items():
            joint_rows.append({'policy':policy,'inputs':group,'targets':'same17','split':split,'joint_windows':int((common&sm).sum())})
    for station in targets:
        valid=past_ok(panel[station])&future_ok(panel[station])
        masks[(policy,'single',station)]=valid
        for split,sm in split_masks.items():
            rows.append({'policy':policy,'inputs':'single','target':station,'split':split,'windows':int((valid&sm).sum())})
window_counts=pd.DataFrame(rows)
joint_counts=pd.DataFrame(joint_rows)
display(joint_counts.pivot(index=['policy','split'],columns='inputs',values='joint_windows'))
display(window_counts.loc[window_counts.split.eq('validation')].pivot(index=['policy','target'],columns='inputs',values='windows'))

inputs                        all18  without_tap_mun17
policy            split                               
category_observed test         8308              12450
                  train       20078              28957
                  validation   6161               8760
exact_only        test         8054              12132
                  train       18783              27593
                  validation   5870               8304

inputs                             all18  single  without_tap_mun17
policy            target                                           
category_observed Causeway Bay      6161    8760               8760
                  Central           6161    8760               8760
                  Central/Western   6161    8760               8760
                  Eastern           6161    8760               8760
                  Kwai Chung        6161    8760               8760
                  Kwun Tong         6161    8760               8760
                  Mong Kok          6161    8760               8760
                  North             6161    8760               8760
                  Sha Tin           6161    8760               8760
                  Sham Shui Po      6161    8760               8760
                  Southern          6161    8760               8760
                  Tai Po            6161    8760               8760
                  Tseung Kwan O     6161    8760               8760
                  Tsuen Wan         6161    8760               8760
                  Tuen Mun          6161    8760               8760
                  Tung Chung        6161    8760               8760
                  Yuen Long         6161    8760               8760
exact_only        Causeway Bay      5930    8760               8400
                  Central           5930    8760               8400
                  Central/Western   5930    8664               8376
                  Eastern           5930    8760               8400
                  Kwai Chung        5930    8760               8400
                  Kwun Tong         5930    8760               8400
                  Mong Kok          5930    8760               8400
                  North             5930    8719               8400
                  Sha Tin           5930    8760               8400
                  Sham Shui Po      5930    8760               8400
                  Southern          5930    8760               8400
                  Tai Po            5930    8718               8400
                  Tseung Kwan O     5930    8760               8400
                  Tsuen Wan         5930    8760               8400
                  Tuen Mun          5870    8401               8329
                  Tung Chung        5895    8567               8353
                  Yuen Long         5907    8623               8377

## 3. 公平比较、缺口与边界测试

模型比较必须使用相同目标和相同预测时点。下面保存三种输入方案共有的训练/验证预测起点清单。
重叠窗口并非独立样本；不能随机拆分窗口。目标区间不得跨划分边界。
测试集只统计可用性，不在本阶段导出用于选模型的比较起点。


In [4]:
gap_start=pd.Timestamp('2023-08-25 01:00',tz=tz)
gap_end=pd.Timestamp('2023-09-01 00:00',tz=tz)
touches=(target_start-pd.Timedelta(hours=L)).le(gap_end)&target_end.ge(gap_start)
fair_parts=[]
for policy in panels:
    for station in targets:
        common=masks[(policy,'single',station)]&masks[(policy,'all18',station)]&masks[(policy,'without_tap_mun17',station)]
        assert not (common&touches).any()
        for split in ['train','validation']:
            starts=idx[common&split_masks[split]]
            fair_parts.append(pd.DataFrame({'policy':policy,'target':station,'split':split,'forecast_start':starts}))
for mask in masks.values(): assert not (mask&touches).any()
assert not (split_masks['train']&split_masks['validation']).any()
assert not (split_masks['validation']&split_masks['test']).any()
# 索引100表示目标第一个小时；输入恰好是此前72小时。
example=100
assert len(idx[example-L:example])==L and len(idx[example:example+H])==H
assert idx[example-1]<idx[example]
fair_origins=pd.concat(fair_parts,ignore_index=True)
display(fair_origins.groupby(['policy','split']).size().rename('station_windows').to_frame())
print('缺口排除、目标边界与索引测试通过。')

station_windows
policy            split                      
category_observed train                341710
                  validation           104737
exact_only        train                322952
                  validation           100692

缺口排除、目标边界与索引测试通过。


## 4. 仅训练期的站点关系

Pearson相关系数仅使用双方有精确数值的训练期小时，10+不参与；同时输出配对观测数。
相关性不代表预测增益，更不代表因果关系。验证期小规模实验才用于选择输入方案。


In [5]:
train_values=values.loc[values.index<=train_end]
correlations=train_values.corr(min_periods=1000)
valid=train_values.notna().astype('int64')
pair_counts=valid.T.dot(valid)
display(correlations.round(2))
display(pair_counts)

station_id,Causeway Bay,Central,Central/Western,Eastern,Kwai Chung,Kwun Tong,Mong Kok,North,Sha Tin,Sham Shui Po,Southern,Tai Po,Tap Mun,Tseung Kwan O,Tsuen Wan,Tuen Mun,Tung Chung,Yuen Long
station_id,,,,,,,,,,,,,,,,,,
Causeway Bay,1.00,0.93,0.90,0.89,0.90,0.91,0.92,0.86,0.88,0.91,0.89,0.88,0.79,0.89,0.88,0.88,0.83,0.86
Central,0.93,1.00,0.93,0.90,0.91,0.93,0.93,0.87,0.90,0.93,0.91,0.89,0.77,0.90,0.89,0.89,0.84,0.87
Central/Western,0.90,0.93,1.00,0.93,0.92,0.92,0.91,0.87,0.91,0.94,0.91,0.88,0.77,0.91,0.91,0.89,0.83,0.88
Eastern,0.89,0.90,0.93,1.00,0.89,0.91,0.90,0.85,0.89,0.91,0.90,0.87,0.80,0.91,0.88,0.86,0.81,0.85
Kwai Chung,0.90,0.91,0.92,0.89,1.00,0.91,0.91,0.89,0.92,0.93,0.90,0.90,0.80,0.90,0.93,0.91,0.86,0.89
Kwun Tong,0.91,0.93,0.92,0.91,0.91,1.00,0.91,0.88,0.91,0.94,0.92,0.90,0.78,0.94,0.90,0.88,0.83,0.87
Mong Kok,0.92,0.93,0.91,0.90,0.91,0.91,1.00,0.87,0.89,0.93,0.90,0.88,0.80,0.89,0.89,0.89,0.83,0.87
North,0.86,0.87,0.87,0.85,0.89,0.88,0.87,1.00,0.92,0.88,0.87,0.94,0.80,0.89,0.90,0.91,0.87,0.92
Sha Tin,0.88,0.90,0.91,0.89,0.92,0.91,0.89,0.92,1.00,0.91,0.91,0.93,0.82,0.92,0.93,0.90,0.87,0.90


station_id,Causeway Bay,Central,Central/Western,Eastern,Kwai Chung,Kwun Tong,Mong Kok,North,Sha Tin,Sham Shui Po,Southern,Tai Po,Tap Mun,Tseung Kwan O,Tsuen Wan,Tuen Mun,Tung Chung,Yuen Long
station_id,,,,,,,,,,,,,,,,,,
Causeway Bay,30749,30746,30723,30743,30739,30741,30743,29789,30748,30738,29795,30742,29419,30742,30738,30711,30728,30717
Central,30746,30750,30725,30742,30740,30740,30745,29790,30749,30738,29796,30743,29422,30741,30740,30714,30730,30720
Central/Western,30723,30725,30726,30721,30726,30721,30721,29773,30725,30725,29782,30721,29402,30720,30726,30704,30716,30709
Eastern,30743,30742,30721,30747,30737,30742,30739,29786,30745,30735,29794,30739,29420,30746,30736,30708,30725,30714
Kwai Chung,30739,30740,30726,30737,30745,30735,30738,29788,30744,30740,29793,30740,29415,30736,30743,30714,30730,30719
Kwun Tong,30741,30740,30721,30742,30735,30743,30737,29783,30741,30735,29791,30735,29416,30741,30735,30707,30722,30713
Mong Kok,30743,30745,30721,30739,30738,30737,30747,29787,30747,30737,29792,30740,29417,30738,30738,30710,30728,30716
North,29789,29790,29773,29786,29788,29783,29787,29795,29795,29785,29782,29793,28498,29788,29788,29769,29781,29777
Sha Tin,30748,30749,30725,30745,30744,30741,30747,29795,30755,30741,29797,30748,29424,30746,30744,30715,30734,30722


## 5. 保存设计依据与下一步

当前不是训练集生成器，不标准化、不训练。下一步在训练/验证期比较基准预测和小模型。
分类与回归评估目标不同，不能直接用各自分数比较优劣；若保留精确回归，必须明确其不覆盖10+的局限。
训练期拟合任何标准化、编码与阈值。测试期锁定，最终方案确定后才报告预测成绩。


In [6]:
for name,table in {'coverage':coverage,'risk_counts':risk_counts,'window_counts':window_counts,
 'joint_counts':joint_counts,'training_correlations':correlations,'training_pair_counts':pair_counts}.items():
    table.to_csv(OUT/f'{name}.csv',encoding='utf-8-sig',index=name not in ['window_counts','joint_counts'])
fair_origins.to_csv(OUT/'fair_train_validation_origins.csv.gz',index=False,compression='gzip')
design={'source':str(SOURCE),'lookback':L,'horizon':H,
 'train_end_inclusive':str(train_end),'validation_end_inclusive':str(val_end),
 'test_end_inclusive':str(idx.max()),'boundary_rule':'all target hours within one split; past observed context allowed',
 'gap_policy':'exclude every input/target span touching registered gap; no imputation',
 'substituted_policy':'retain flag and observed value for this feasibility audit',
 'comparison_targets':targets,'training_complete':False,'model_choice':'not selected',
 'test_policy':'availability counts only; do not use test distribution to tune or select models'}
(OUT/'design.json').write_text(json.dumps(design,ensure_ascii=False,indent=2),encoding='utf-8')
print('已保存设计、覆盖统计、训练期相关性与公平比较起点：',OUT)

已保存设计、覆盖统计、训练期相关性与公平比较起点： C:\Users\hxq08259\Desktop\sem1\Math7005\data\processed\design\20261110_180535_638894
